In [1]:
import scanpy as sc
import pandas as pd
import os
import numpy as np
import re

### 预处理
This notebook is largely derived from the preprocessing followed by Lotfollahi, Mohammad, et al. "Predicting cellular responses to complex perturbations in high‐throughput screens." Molecular systems biology 19.6 (2023): e11517.

See https://github.com/facebookresearch/CPA/blob/main/preprocessing/Norman19.ipynb 

我这里直接上ncbi下载文件 参考了GEARS和CPA的预处理notebook

In [2]:
# original data from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE133344  #filtered
adata = sc.read_10x_mtx(
    './raw/norman_filtered_raw',  
    var_names='gene_ids',
    cache=True
)

meta = pd.read_csv('./raw/norman_filtered_raw/cell_identities.csv', index_col=0)

adata.obs = adata.obs.join(meta)
adata.obs['good_coverage'] = adata.obs['good_coverage'].astype(str)

print("转换完成！")

转换完成！


In [3]:
adata

AnnData object with n_obs × n_vars = 111668 × 33694
    obs: 'guide_identity', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells'
    var: 'gene_symbols'

In [4]:
adata.var

,gene_symbols
ENSG00000243485,RP11-34P13.3
ENSG00000237613,FAM138A
ENSG00000186092,OR4F5
ENSG00000238009,RP11-34P13.7
ENSG00000239945,RP11-34P13.8
...,...
ENSG00000277856,AC233755.2
ENSG00000275063,AC233755.1
ENSG00000271254,AC240274.1
ENSG00000277475,AC213203.1


In [5]:
adata.obs

,guide_identity,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells
AAACCTGAGAAGAAGC-1,NegCtrl0_NegCtrl0__NegCtrl0_NegCtrl0,1252.0,67.0,18.686567,1.0,True,2.0
AAACCTGAGGCATGTG-1,TSC22D1_NegCtrl0__TSC22D1_NegCtrl0,2151.0,104.0,20.682692,1.0,True,1.0
AAACCTGAGGCCCTTG-1,KLF1_MAP2K6__KLF1_MAP2K6,1037.0,59.0,17.576271,1.0,True,1.0
AAACCTGCACGAAGCA-1,NegCtrl10_NegCtrl0__NegCtrl10_NegCtrl0,958.0,39.0,24.564103,1.0,True,1.0
AAACCTGCAGACGTAG-1,CEBPE_RUNX1T1__CEBPE_RUNX1T1,244.0,14.0,17.428571,1.0,True,1.0
...,...,...,...,...,...,...,...
TTTGTCATCAGTACGT-8,FOXA3_NegCtrl0__FOXA3_NegCtrl0,2068.0,95.0,21.768421,8.0,True,1.0
TTTGTCATCCACTCCA-8,CELF2_NegCtrl0__CELF2_NegCtrl0,829.0,33.0,25.121212,8.0,True,1.0
TTTGTCATCCCAACGG-8,BCORL1_NegCtrl0__BCORL1_NegCtrl0,136.0,9.0,15.111111,8.0,True,1.0
TTTGTCATCCTCCTAG-8,ZBTB10_PTPN12__ZBTB10_PTPN12,1254.0,59.0,21.254237,8.0,True,3.0


In [6]:
# remove "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0" suggested by authors
adata = adata[adata.obs["guide_identity"] != "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0"] 
# reserve only good_coverage == True 原文没有 但感觉得这么干
adata = adata[adata.obs["good_coverage"]=="True"]

In [7]:
needed_obs = adata.obs[["guide_identity", "UMI_count","gemgroup","number_of_cells"]].copy()
adata_new = sc.AnnData(adata.X.copy(), obs=needed_obs, var=adata.var.copy())

In [8]:
adata_new

AnnData object with n_obs × n_vars = 101719 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells'
    var: 'gene_symbols'

## prepare data

In [11]:
# merge control
adata_new.obs["guide_merged"] = adata_new.obs["guide_identity"].astype(str)
for i in np.unique(adata_new.obs["guide_merged"]):
   m = re.match(r"NegCtrl(.*)_NegCtrl(.*)__NegCtrl(.*)_NegCtrl(.*)", i)
   if m :
        adata_new.obs["guide_merged"].replace(i,"ctrl",inplace=True)

In [12]:
# relabel
old_pool = []
for i in np.unique(adata_new.obs["guide_merged"]):
    if i == "ctrl":
        old_pool.append(i)
        continue
    split = i.split("__")[1]
    split = split.split("_")
    for j, string in enumerate(split):
        if "NegCtrl" in split[j]:
            split[j] = "ctrl"
    if len(split) == 1:
        if split[0] in old_pool:
            print("old:",i, "new:",split[0])
        adata_new.obs["guide_merged"].replace(i,split[0],inplace=True)
        old_pool.append(split[0])
    else:
        if f"{split[0]}+{split[1]}" in old_pool:
            print("old:",i, "new:",f"{split[0]}+{split[1]}")
        adata_new.obs["guide_merged"].replace(i, f"{split[0]}+{split[1]}",inplace=True)
        old_pool.append(f"{split[0]}+{split[1]}")

old: HOXC13_NegCtrl0__HOXC13_NegCtrl0_2 new: HOXC13+ctrl
old: TGFBR2_IGDCC3__TGFBR2_IGDCC3_2 new: TGFBR2+IGDCC3
old: ZBTB10_NegCtrl0__ZBTB10_NegCtrl0_2 new: ZBTB10+ctrl


In [16]:
adata_new.obs["guide_merged"].value_counts()

guide_merged
ctrl             8395
CEBPE+RUNX1T1    1135
KLF1+ctrl        1120
TBX3+TBX2        1094
SLC4A1+ctrl       937
                 ... 
CBL+UBASH3A        56
C3orf72+FOXL2      55
CEBPB+CEBPA        55
JUN+CEBPB          54
JUN+CEBPA          51
Name: count, Length: 284, dtype: int64

In [17]:
adata_new.write("./raw/my_norman.h5ad")

In [2]:
adata_new = sc.read_h5ad("./raw/my_norman.h5ad")

In [5]:
adata_new

AnnData object with n_obs × n_vars = 101719 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged'
    var: 'gene_symbols'

In [8]:
conditions = [(c.split('+')[0], c.split('+')[1]) for c in adata_new.obs['guide_merged'] if '+' in c]
conditions = [item for sublist in conditions for item in sublist]
genes_to_keep = np.unique(conditions)
genes_to_keep = genes_to_keep[genes_to_keep!="ctrl"]

In [9]:
genes_to_keep

array(['AHR', 'ARID1A', 'ARRDC3', 'ATL1', 'BAK1', 'BCL2L11', 'BCORL1',
       'BPGM', 'C19orf26', 'C3orf72', 'CBFA2T3', 'CBL', 'CDKN1A',
       'CDKN1B', 'CDKN1C', 'CEBPA', 'CEBPB', 'CEBPE', 'CELF2', 'CITED1',
       'CKS1B', 'CLDN6', 'CNN1', 'CNNM4', 'COL1A1', 'COL2A1', 'CSRNP1',
       'DLX2', 'DUSP9', 'EGR1', 'ELMSAN1', 'ETS2', 'FEV', 'FOSB', 'FOXA1',
       'FOXA3', 'FOXF1', 'FOXL2', 'FOXO4', 'GLB1L2', 'HES7', 'HK2',
       'HNF4A', 'HOXA13', 'HOXB9', 'HOXC13', 'IER5L', 'IGDCC3', 'IKZF3',
       'IRF1', 'ISL2', 'JUN', 'KIAA1804', 'KIF18B', 'KIF2C', 'KLF1',
       'KMT2A', 'LHX1', 'LYL1', 'MAML2', 'MAP2K3', 'MAP2K6', 'MAP4K3',
       'MAP4K5', 'MAP7D1', 'MAPK1', 'MEIS1', 'MIDN', 'NCL', 'NIT1',
       'OSR2', 'PLK4', 'POU3F2', 'PRDM1', 'PRTG', 'PTPN1', 'PTPN12',
       'PTPN13', 'PTPN9', 'RHOXF2', 'RREB1', 'RUNX1T1', 'S1PR2', 'SAMD1',
       'SET', 'SGK1', 'SLC38A2', 'SLC4A1', 'SLC6A9', 'SNAI1', 'SPI1',
       'STIL', 'TBX2', 'TBX3', 'TGFBR2', 'TMSB4X', 'TP73', 'TSC22D1',
       'UBA

In [13]:
df = pd.DataFrame(genes_to_keep, columns=['gene'])
df.to_csv('target_genes_norman.csv', index=False)